# Getting Started with HydroClaude

**Version**: 1.0  
**Date**: 2025-10-29  
**Level**: Beginner

## Welcome! 🌊

This notebook provides an interactive introduction to HydroClaude, a 1D open channel flow solver. You'll learn how to:

1. Set up a basic simulation
2. Run the solver
3. Visualize results
4. Understand the physics

### Prerequisites

- Basic Python knowledge
- Understanding of open channel hydraulics (helpful but not required)

Let's get started!

## Step 1: Import Libraries

First, let's import the necessary libraries:

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to path to import HydroClaude
sys.path.insert(0, '..')

from engine.model_builder import ModelBuilder

# Set up matplotlib for nice plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries imported successfully!")

## Step 2: Define Your First Simulation

Let's simulate flow in a simple rectangular channel. We'll use the **Godunov Finite Volume Method**, which is robust and widely used for open channel flow.

### Physical Setup

- **Channel**: 1000m long, 10m wide
- **Slope**: 0.001 (mild slope)
- **Manning's n**: 0.03 (natural earth channel)
- **Flow**: 20 m³/s discharge
- **Initial depth**: 1.5m

In [ ]:
# Define configuration
config = {
    'geometry': {
        'channel_width': 10.0,      # Channel width (m)
        'channel_length': 1000.0,   # Channel length (m)
        'manning_n': 0.03,          # Manning's roughness coefficient
        'bed_slope': 0.001          # Channel slope (m/m)
    },
    'mesh': {
        'n_cells': 100              # Number of computational cells
    },
    'solver': {
        'type': 'godunov_fvm',      # Solver type
        'spatial_order': 1,          # 1st order (robust)
        'riemann_solver': 'hll',     # HLL Riemann solver
        'cfl': 0.5,                  # CFL number
        'dt_max': 0.5,               # Maximum time step (s)
        'eps_dry': 1e-6,             # Dry bed threshold
        'use_numba': True,           # Enable fast computation
        'well_balanced': False
    },
    'initial_conditions': {
        'h0': 1.5,                   # Initial water depth (m)
        'Q0': 20.0                   # Initial discharge (m³/s)
    },
    'boundary_conditions': {
        'left': {'type': 'Q', 'value': 20.0},    # Inflow: 20 m³/s
        'right': {'type': 'h', 'value': 2.0}     # Outflow: 2m depth
    }
}

print("✅ Configuration defined!")
print(f"   Channel: {config['geometry']['channel_length']}m × {config['geometry']['channel_width']}m")
print(f"   Grid: {config['mesh']['n_cells']} cells")
print(f"   Flow rate: {config['initial_conditions']['Q0']} m³/s")

## Step 3: Build and Initialize the Model

Now let's create the solver using the `ModelBuilder`:

In [ ]:
# Build the model
builder = ModelBuilder(config)
solver = builder.solver

print("✅ Model built successfully!")
print(f"   Solver: {solver.__class__.__name__}")
print(f"   Domain: {solver.length:.1f}m, dx={solver.dx:.1f}m")
print(f"   Initial CFL dt: {solver.compute_dt():.3f}s")

## Step 4: Run the Simulation

Let's run the simulation for 500 seconds and save results every 50 seconds:

In [ ]:
# Simulation parameters
t_end = 500.0       # End time (s)
dt_output = 50.0    # Output interval (s)

# Storage for results
results = {
    't': [],
    'h': [],
    'Q': [],
    'x': solver.x.copy()
}

# Time stepping
t = 0.0
t_next_output = 0.0
step = 0

print("🚀 Starting simulation...")
print(f"   End time: {t_end}s")
print(f"   Output every: {dt_output}s\n")

while t < t_end:
    # Compute adaptive time step
    dt = solver.compute_dt()
    
    # Take one time step
    solver.step(dt)
    t += dt
    step += 1
    
    # Save output
    if t >= t_next_output:
        results['t'].append(t)
        results['h'].append(solver.h.copy())
        results['Q'].append(solver.Q.copy())
        
        h_avg = solver.h.mean()
        Q_avg = solver.Q.mean()
        print(f"   t={t:6.1f}s (step {step:5d}): h_avg={h_avg:.3f}m, Q_avg={Q_avg:.2f} m³/s, dt={dt:.3f}s")
        
        t_next_output += dt_output

print(f"\n✅ Simulation complete!")
print(f"   Total steps: {step}")
print(f"   Average dt: {t_end/step:.3f}s")

## Step 5: Visualize the Results

### 5.1 Water Depth Evolution

In [ ]:
# Plot water depth profiles at different times
fig, ax = plt.subplots(figsize=(12, 6))

x = results['x']
colors = plt.cm.viridis(np.linspace(0, 1, len(results['t'])))

for i, (t_val, h) in enumerate(zip(results['t'], results['h'])):
    ax.plot(x, h, label=f't={t_val:.0f}s', color=colors[i], linewidth=2)

ax.set_xlabel('Distance (m)', fontsize=12)
ax.set_ylabel('Water Depth (m)', fontsize=12)
ax.set_title('Water Depth Evolution', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Water depth profile shows flow approaching steady state")

### 5.2 Final State Analysis

In [ ]:
# Analyze final state
h_final = results['h'][-1]
Q_final = results['Q'][-1]
x = results['x']

# Compute flow properties
B = solver.B
A = h_final * B
u = Q_final / A
Fr = u / np.sqrt(9.81 * h_final)  # Froude number

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Water depth
axes[0, 0].plot(x, h_final, 'b-', linewidth=2)
axes[0, 0].set_xlabel('Distance (m)')
axes[0, 0].set_ylabel('Depth h (m)')
axes[0, 0].set_title('Water Depth Profile')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Discharge
axes[0, 1].plot(x, Q_final, 'g-', linewidth=2)
axes[0, 1].set_xlabel('Distance (m)')
axes[0, 1].set_ylabel('Discharge Q (m³/s)')
axes[0, 1].set_title('Discharge Profile')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Velocity
axes[1, 0].plot(x, u, 'r-', linewidth=2)
axes[1, 0].set_xlabel('Distance (m)')
axes[1, 0].set_ylabel('Velocity u (m/s)')
axes[1, 0].set_title('Flow Velocity')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Froude number
axes[1, 1].plot(x, Fr, 'm-', linewidth=2)
axes[1, 1].axhline(y=1.0, color='k', linestyle='--', label='Critical (Fr=1)')
axes[1, 1].fill_between(x, 0, 1, alpha=0.2, color='blue', label='Subcritical')
axes[1, 1].fill_between(x, 1, Fr.max(), alpha=0.2, color='red', label='Supercritical')
axes[1, 1].set_xlabel('Distance (m)')
axes[1, 1].set_ylabel('Froude Number')
axes[1, 1].set_title('Flow Regime')
axes[1, 1].legend(loc='best')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("\n📈 Flow Statistics:")
print(f"   Average depth: {h_final.mean():.3f}m (range: {h_final.min():.3f}-{h_final.max():.3f}m)")
print(f"   Average discharge: {Q_final.mean():.2f} m³/s")
print(f"   Average velocity: {u.mean():.3f} m/s (range: {u.min():.3f}-{u.max():.3f} m/s)")
print(f"   Average Froude: {Fr.mean():.3f} (range: {Fr.min():.3f}-{Fr.max():.3f})")
print(f"   Flow regime: {'Subcritical' if Fr.mean() < 1 else 'Supercritical'}")

### 5.3 Mass Conservation Check

One of the most important properties of a good numerical scheme is mass conservation:

In [ ]:
# Calculate mass at different times
masses = []
for h in results['h']:
    mass = np.sum(h * B * solver.dx)  # Total water volume
    masses.append(mass)

mass_initial = masses[0]
mass_final = masses[-1]
mass_error = abs(mass_final - mass_initial) / mass_initial * 100

# Plot mass evolution
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(results['t'], masses, 'b-', linewidth=2, marker='o')
ax.axhline(y=mass_initial, color='r', linestyle='--', label=f'Initial mass: {mass_initial:.1f} m³')
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Total Water Volume (m³)', fontsize=12)
ax.set_title('Mass Conservation Check', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💧 Mass Conservation:")
print(f"   Initial mass: {mass_initial:.2f} m³")
print(f"   Final mass: {mass_final:.2f} m³")
print(f"   Mass error: {mass_error:.2f}%")

if mass_error < 2:
    print("   ✅ Excellent conservation!")
elif mass_error < 5:
    print("   ✅ Good conservation")
elif mass_error < 10:
    print("   ⚠️  Acceptable conservation")
else:
    print("   ❌ Poor conservation - check setup")

## Step 6: Understanding the Physics

### 6.1 Normal Depth

For uniform flow, we can calculate the theoretical **normal depth** using Manning's equation:

In [ ]:
# Manning's equation for normal depth
# Q = (1/n) * A * R^(2/3) * S^(1/2)
# For rectangular channel: R ≈ h (wide channel), A = B*h
# Solving: h_n = (Q*n / (B*sqrt(S)))^(3/5)

Q = 20.0
B = 10.0
n = 0.03
S = 0.001

h_normal = (Q * n / (B * np.sqrt(S))) ** (3/5)

print("\n📐 Theoretical Analysis:")
print(f"   Normal depth (Manning): {h_normal:.3f}m")
print(f"   Simulated average depth: {h_final.mean():.3f}m")
print(f"   Deviation: {abs(h_final.mean() - h_normal) / h_normal * 100:.1f}%")

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, h_final, 'b-', linewidth=2, label='Simulated')
ax.axhline(y=h_normal, color='r', linestyle='--', linewidth=2, label=f'Normal depth: {h_normal:.3f}m')
ax.fill_between(x, h_normal*0.95, h_normal*1.05, alpha=0.2, color='green', label='±5% band')
ax.set_xlabel('Distance (m)', fontsize=12)
ax.set_ylabel('Water Depth (m)', fontsize=12)
ax.set_title('Comparison with Normal Depth', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.2 Critical Flow and Froude Number

The **Froude number** (Fr) characterizes the flow regime:

- **Fr < 1**: Subcritical flow (slow, deep)
- **Fr = 1**: Critical flow (unstable)
- **Fr > 1**: Supercritical flow (fast, shallow)

For rectangular channels, critical depth is:

In [ ]:
# Critical depth: h_c = (Q²/(g*B²))^(1/3)
g = 9.81
q = Q / B  # Unit discharge
h_critical = (q**2 / g) ** (1/3)

print("\n🌊 Flow Regimes:")
print(f"   Critical depth: {h_critical:.3f}m")
print(f"   Normal depth: {h_normal:.3f}m")
print(f"   Simulated depth: {h_final.mean():.3f}m")
print(f"\n   Since h_normal ({h_normal:.3f}m) > h_critical ({h_critical:.3f}m):")
print(f"   → Flow is SUBCRITICAL")
print(f"   → Average Froude number: {Fr.mean():.3f} < 1 ✓")

## Summary and Next Steps

Congratulations! 🎉 You've successfully:

✅ Configured a 1D open channel flow simulation  
✅ Run the Godunov FVM solver  
✅ Visualized water depth, discharge, velocity, and Froude number  
✅ Verified mass conservation  
✅ Compared results with theoretical predictions  

### What's Next?

1. **Experiment with parameters**:
   - Try different slopes, Manning's n, flow rates
   - Change grid resolution (`n_cells`)
   - Adjust time step limits (`dt_max`)

2. **Try other example notebooks**:
   - `02_dam_break.ipynb` - Transient flow simulation
   - `03_backwater_curve.ipynb` - Subcritical flow profiles
   - `04_supercritical_flow.ipynb` - Fast flow regime

3. **Explore advanced features**:
   - Boundary condition types
   - Manning friction effects
   - Spatial accuracy (1st vs 2nd order)

4. **Read the documentation**:
   - `docs/GODUNOV_FVM_USER_GUIDE.md` - Complete solver guide
   - `docs/PROJECT_STATUS_2025_10_29.md` - Project overview
   - `tests/standard_tests/test_macdonald.py` - Validation cases

### Feedback

Found this notebook helpful? Have suggestions? Please let us know!

Happy hydraulic modeling! 🌊💧